In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pytorch import *

/data/vision/beery/scratch/justin/movebench/pytorch.py:38: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  modis_grid = pd.read_csv(buf, delim_whitespace=True, header=None, names=cols)


In [3]:
from torch.utils.data import DataLoader

In [4]:
study = '5a280544-06db-4c85-9713-842547972edd' # '338388ea-6483-472c-9cdc-c2997e645df0'
df = pd.read_csv(glob.glob(f'mbr_studies_2year_filtered_traintest/{study}*')[0])
df = df[df.set == 'train']

In [5]:
# import time
ds = ChipDataset(study, idx_to_timestamp=pd.to_datetime(df.timestamp))
ds[393][0]

tensor([[[-8.3843e-01, -8.3843e-01, -8.3843e-01,  ..., -7.4757e-01,
          -7.4757e-01,        -inf],
         [-1.4432e+00, -1.4432e+00, -1.4432e+00,  ...,        -inf,
                 -inf,        -inf],
         [-1.4432e+00, -1.4432e+00, -1.4432e+00,  ...,        -inf,
                 -inf,        -inf],
         ...,
         [       -inf,        -inf,        -inf,  ...,  5.8403e-01,
           2.4918e-01,  2.4918e-01],
         [       -inf,        -inf,        -inf,  ...,  5.8403e-01,
           2.4918e-01,  2.4918e-01],
         [       -inf,        -inf,        -inf,  ...,  5.8403e-01,
           5.8403e-01,  5.8403e-01]],

        [[-3.2679e-01, -3.2679e-01, -3.2679e-01,  ..., -8.1800e-01,
          -8.1800e-01,        -inf],
         [-2.7483e-01, -2.7483e-01, -2.7483e-01,  ...,        -inf,
                 -inf,        -inf],
         [-2.7483e-01, -2.7483e-01, -2.7483e-01,  ...,        -inf,
                 -inf,        -inf],
         ...,
         [-1.3785e-01, -1

------

Standardization

In [ ]:
## Standardization ##
## TODO: Subsample ~1000 randomly per study ##
from tqdm import tqdm

studies = [ d for d in os.listdir('dem_out') if os.path.isdir('dem_out/'+d) ]
studies.remove('c73c7698-2c6b-44d4-8879-fe10d7536935') # this one failed

for study in studies:
    if os.path.exists(f'{study}_mean.pt'):
        print("Already computed", study)
    else:
        print("Getting standardization values for", study)

        df = pd.read_csv(glob.glob(f'mbr_studies_2year_filtered_traintest/{study}*')[0])
        df = df[df.set == 'train']
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
        df = df.dropna(subset=['timestamp'])
        
        ds = ChipDataset(study, idx_to_timestamp=pd.to_datetime(df.timestamp))
        dataloader = DataLoader(ds, 
                        batch_size=1,
                        shuffle=True,
                        num_workers=16)

        # get values for standardization
        layer_sum = None
        layer_sq_sum = None
        count = None
        num_to_sample = 1000
        for i_batch, (x, valid_mask, _, _) in tqdm(enumerate(dataloader), total=num_to_sample):
            x = x.squeeze(0)
            valid_mask = valid_mask.squeeze(0)
            num_valid  = valid_mask.sum(dim=(-1, -2))  # (C,)
            x_clean = torch.where(valid_mask, x, torch.zeros_like(x))

            # Per-layer sums
            s  = x_clean.sum(dim=(-1, -2))         # (C,)
            s2 = (x_clean ** 2).sum(dim=(-1, -2))  # (C,)

            if layer_sum is None:
                layer_sum = s
                layer_sq_sum = s2
                count = num_valid
            else:
                layer_sum += s
                layer_sq_sum += s2
                count += num_valid

            # just sample 1000 chips max
            if i_batch >= num_to_sample: 
                break

        print("Done")

        mean = layer_sum / count
        var  = layer_sq_sum / count - mean**2
        var = torch.clamp(var, min=0.0)
        std = torch.sqrt(var)

        torch.save(mean, f'{study}_mean.pt')
        torch.save(std,  f'{study}_std.pt')

        print("Saved", study)

----

SSF

In [6]:
ssf_df = setup_df_for_ssf(df)
ssf_df.head()

/data/vision/beery/scratch/justin/movebench/pytorch.py:413: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_step_next = df['valid_step'].shift(-1).fillna(False)  # step from current -> next


,Unnamed: 0,event-id,visible,timestamp,location-long,location-lat,gps:fix-type,gps:fix-type-raw,gps:hdop,gps:satellite-count,...,within_time,step_length_m,valid_step,max_step_len,bearing,valid_step_prev,turn_angle_rad,burst_id,next_index,can_be_ssf_start
0,0,29538063317,True,2019-04-10 00:00:00,-92.823760,57.877814,2,Resolved QFP,1.5,9,...,False,NaN,False,6357.680811,NaN,NaN,NaN,1,1,False
1,1,29538063318,True,2019-04-10 01:30:00,-92.824403,57.877886,2,Resolved QFP,6.1,6,...,True,38.851370,True,6357.680811,-1.363236,False,NaN,1,2,True
2,2,29538063319,True,2019-04-10 03:00:00,-92.859990,57.887560,2,Resolved QFP,1.5,7,...,True,2362.863555,True,6357.680811,-1.097877,True,0.265359,1,3,True
3,3,29538063320,True,2019-04-10 04:30:00,-92.885635,57.887463,2,Resolved QFP,1.1,8,...,True,1515.897800,True,6357.680811,-1.577722,True,-0.479845,1,4,True
4,4,29538063321,True,2019-04-10 06:00:00,-92.885662,57.887452,2,Resolved QFP,1.1,8,...,True,2.010760,True,6357.680811,-2.224713,True,-0.646991,1,5,True


In [9]:
sds = SSFDataset(study, df)
step_pixels, valid_pixels, chip_transform, chip_crs = sds[0]
step_pixels.shape, valid_pixels.shape

/data/vision/beery/scratch/justin/movebench/pytorch.py:413: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_step_next = df['valid_step'].shift(-1).fillna(False)  # step from current -> next


norm mask shape torch.Size([63])
interp pixels shape torch.Size([63, 101])
interp pixels shape torch.Size([63, 101])


(torch.Size([63, 101]), torch.Size([63, 101]))

In [12]:
step_pixels[:,0], valid_pixels[:,0]

(tensor([-1.3134, -2.5892, -0.7773, -0.8278,  0.4991, -0.3273,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf, -0.8631, -0.1525,  0.8042, -0.5001,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         -0.3064,  0.0054, -0.3894,  0.1823, -0.2177,  0.1173,  0.1864,  0.0585,
         -0.0510, -0.1159,  0.0519, -0.0471, -0.1121, -0.0907, -0.1141,  0.1403,
          0.1798,  0.0737,  0.1087,  0.1330,  0.1059,  0.1366,  0.2060]),
 tensor([ True,  True,  True,  True,  True,  True, False, False, False, False,
         False, False, False, False, False, False, False, False,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,

In [ ]:
rng = np.random.default_rng(0)
steps,turns = sds._get_pseudo_steps_and_turns(100, rng)

----

Compute and cache features


In [ ]:
from tqdm import tqdm

studies = [ d for d in os.listdir('dem_out') if os.path.isdir('dem_out/'+d) ]
studies.remove('c73c7698-2c6b-44d4-8879-fe10d7536935') # this one failed

for study in studies:
    for SET in ['train', 'test']:
        # if os.path.exists(f'{study}_{SET}_xs.pt'):
        #     print("Already computed SSF features for", study, SET)
        # else:
            print("Getting SSF features values for", study, SET)
            df = pd.read_csv(glob.glob(f'mbr_studies_2year_filtered_traintest/{study}*')[0])
            df = df[df.set == SET]
            sds = SSFDataset(study, df)

            dataloader = DataLoader(sds, 
                            batch_size=1,
                            shuffle=True,
                            num_workers=64,
                            # multiprocessing_context='spawn', # need to move to a separate module for this to work - needs to be picklable
                            )
            all_xs = []
            all_masks = []
            all_ys = []
            for i_batch, (step_pixels, valid_pixels, _, _) in tqdm(enumerate(dataloader), total=len(sds)):
                all_xs.append(step_pixels)
                all_masks.append(valid_pixels)
                all_ys.append(torch.tensor( [0]*sds.num_candidates + [1] ))

            all_xs_t    = torch.cat(all_xs).permute(0, 2, 1).reshape(-1, 63)
            all_masks_t = torch.cat(all_masks).permute(0, 2, 1).reshape(-1, 63)
            all_ys_t    = torch.cat(all_ys)

            torch.save(all_xs_t, f'{study}_{SET}_xs.pt')
            torch.save(all_masks_t, f'{study}_{SET}_masks.pt')
            torch.save(all_ys_t, f'{study}_{SET}_ys.pt')

-----

Train ssf

In [48]:
import torch

In [10]:
# study = studies[0]
study = 'fd28aac2-b22f-4368-8638-59906e769ded'
all_xs_t = torch.load(f'{study}_train_xs.pt')
all_masks_t = torch.load(f'{study}_train_masks.pt')
all_ys_t = torch.load(f'{study}_train_ys.pt')

all_xs_t.shape, all_masks_t.shape, all_ys_t.shape

(torch.Size([409252, 63]), torch.Size([409252, 63]), torch.Size([409252]))

In [12]:
X_tr = all_xs_t
masks_tr = all_masks_t
y_tr = all_ys_t

starts_tr = torch.arange(0, X_tr.shape[0], 101)
ends_tr = torch.arange(101, X_tr.shape[0]+1, 101)

# fill invalid values with (approximately) the mean (0 after standardization)
X_tr = torch.where(masks_tr, X_tr, torch.tensor(0.0, device=X_tr.device))

# and hack the infs for now...
X_tr = torch.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)


# test data
all_xs_test = torch.load(f'{study}_test_xs.pt')
all_masks_test = torch.load(f'{study}_test_masks.pt')
all_ys_test = torch.load(f'{study}_test_ys.pt')
all_xs_test.shape, all_masks_test.shape, all_ys_test.shape
X_te = all_xs_test
masks_te = all_masks_test
y_te = all_ys_test
starts_te = torch.arange(0, X_te.shape[0], 101)
ends_te = torch.arange(101, X_te.shape[0]+1, 101)
X_te = torch.where(masks_te, X_te, torch.tensor(0.0, device=X_te.device))
X_te = torch.nan_to_num(X_te, nan=0.0, posinf=0.0, neginf=0.0) # hack

In [13]:
p = all_xs_t.shape[1]
beta = torch.zeros(p, dtype=torch.float32, requires_grad=True)
opt = torch.optim.Adam([beta], lr=1e-2, weight_decay=1e-3)  # weight_decay = L2

for step in range(1000):
    opt.zero_grad()
    loss = grouped_nll(beta, X_tr, y_tr, starts_tr, ends_tr)
    loss.backward()
    opt.step()

    if step % 50 == 0:
        with torch.no_grad():
            nll_tr = grouped_nll(beta, X_tr, y_tr, starts_tr, ends_tr).item()
            nll_te = grouped_nll(beta, X_te, y_te, starts_te, ends_te).item()
            print(step, "NLL train", nll_tr, "NLL test", nll_te)

0 NLL train 4.612761974334717 NLL test 4.612614154815674
50 NLL train 4.592836380004883 NLL test 4.595242023468018
100 NLL train 4.591442108154297 NLL test 4.594956398010254
150 NLL train 4.591264724731445 NLL test 4.595198154449463
200 NLL train 4.5912299156188965 NLL test 4.595259666442871
250 NLL train 4.591217041015625 NLL test 4.5952606201171875
300 NLL train 4.591212272644043 NLL test 4.595272064208984
350 NLL train 4.59121561050415 NLL test 4.595276832580566
400 NLL train 4.591212272644043 NLL test 4.595273494720459
450 NLL train 4.591215133666992 NLL test 4.595274448394775
500 NLL train 4.591214179992676 NLL test 4.595274925231934
550 NLL train 4.591214179992676 NLL test 4.595274925231934
600 NLL train 4.591214179992676 NLL test 4.595274925231934
650 NLL train 4.591214179992676 NLL test 4.595274925231934
700 NLL train 4.591214179992676 NLL test 4.595274925231934
750 NLL train 4.591214179992676 NLL test 4.595274925231934
800 NLL train 4.591214656829834 NLL test 4.595274925231934

In [11]:
# with LBFGS

# p = all_xs_t.shape[1]
# beta = torch.zeros(p, dtype=torch.float32, requires_grad=True)
# opt = torch.optim.LBFGS([beta], max_iter=10000, line_search_fn="strong_wolfe")

# def closure():
#     opt.zero_grad()
#     loss = grouped_nll(beta, X_tr, y_tr, starts_tr, ends_tr)
#     loss.backward()

#     print("NLL train", loss)
#     with torch.no_grad():
#         nll_te = grouped_nll(beta, X_te, y_te, starts_te, ends_te).item()
#         print("NLL test:", nll_te)
    
#     return loss

# print("Fitting...")
# opt.step(closure)
# print("Done.")

# print("beta", beta)

# # Train / Test NLL
# with torch.no_grad():
#     nll_tr = grouped_nll(beta, X_tr, y_tr, starts_tr, ends_tr).item()
# print("NLL train:", nll_tr)

# print("beta", beta)

# # Back-transform coefficients to original feature units if you want to compare to R
# # beta_hat_orig = (beta.detach().cpu().numpy()) / sd
# # for name, val in zip(cols, beta_hat_orig):
# #     print(name, val)
